# Βήμα 5 – Πείραμα: επίδραση των βαρών κλάσεων (Σύστημα Α)
Το βάρος κάθε κλάσης στο σφάλμα είναι ανάλογο του `1 / συχνότητα^p`:
- `p = 0` → χωρίς βάρη (όλες οι κλάσεις ίδιο βάρος)
- `p = 0.25` → ήπια βάρη
- `p = 0.5` → τα βάρη της πρώτης εκπαίδευσης

Εκπαιδεύουμε το ίδιο δίκτυο τρεις φορές και συγκρίνουμε. **Η επιλογή γίνεται με βάση το validation**· το test το βλέπουμε μόνο για την τελική αναφορά.
Χρόνος: περίπου τριπλάσιος της πρώτης εκπαίδευσης.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

DATA, MODELS, FIG = "../data/", "../models/", "../figures/"
SEED = 42
POWERS = [0.0, 0.25, 0.5]

d = np.load(DATA + "unsw_prepared.npz")
with open(MODELS + "unsw_meta.json", encoding="utf-8") as f:
    meta = json.load(f)
ATTACK_CLASSES, SEVERITY_CLASSES = meta["attack_classes"], meta["severity_classes"]
SEV_OF_ATTACK = np.array([SEVERITY_CLASSES.index(meta["severity_map"][c]) for c in ATTACK_CLASSES])

X = {s: d[f"X_{s}"] for s in ("train", "val", "test")}
y = {s: {k: d[f"y_{s}_{k}"].astype(int) for k in ("label", "attack_cat", "severity")} for s in ("train", "val", "test")}

def targets(s):
    return [y[s]["label"].astype("float32").reshape(-1, 1), y[s]["attack_cat"].astype("int32"), y[s]["severity"].astype("int32")]

## 5.1 Βάρη, δίκτυο και κανόνας απόφασης (ίδια με πριν, με παράμετρο `p`)

In [ ]:
def class_weights(y_cls, n_classes, power):
    counts = np.bincount(y_cls, minlength=n_classes).astype("float64")
    w = (len(y_cls) / (n_classes * np.maximum(counts, 1))) ** power
    return (w / np.average(w, weights=counts)).astype("float32")      # μέσο βάρος = 1

def weighted_sparse_ce(class_w):
    w = keras.ops.convert_to_tensor(class_w, dtype="float32")
    def loss(y_true, y_pred):
        yy = keras.ops.cast(keras.ops.reshape(y_true, (-1,)), "int32")
        return keras.losses.sparse_categorical_crossentropy(yy, y_pred) * keras.ops.take(w, yy)
    return loss

def build_model(n_features, w_attack, w_sev, units=(128, 64, 32), dropout=0.3, lr=1e-3):
    inp = keras.Input(shape=(n_features,), name="flow_features")
    x = inp
    for i, u in enumerate(units):
        x = layers.Dense(u, activation="relu", name=f"dense_{i+1}")(x)
        x = layers.BatchNormalization(name=f"bn_{i+1}")(x)
        x = layers.Dropout(dropout, name=f"dropout_{i+1}")(x)
    outs = [layers.Dense(1, activation="sigmoid", name="label")(x),
            layers.Dense(len(ATTACK_CLASSES), activation="softmax", name="attack_cat")(x),
            layers.Dense(len(SEVERITY_CLASSES), activation="softmax", name="severity")(x)]
    model = keras.Model(inp, outs, name="unsw_multitask")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss=["binary_crossentropy", weighted_sparse_ce(w_attack), weighted_sparse_ce(w_sev)],
                  loss_weights=[1.0, 1.0, 0.5])
    return model

def decide(p_label, p_attack, threshold=0.5):
    is_attack = p_label >= threshold
    attack = np.where(is_attack, p_attack[:, 1:].argmax(1) + 1, 0)
    return is_attack.astype(int), attack, SEV_OF_ATTACK[attack]

def evaluate(model, split):
    p_label, p_attack, _ = model.predict(X[split], batch_size=1024, verbose=0)
    lab, att, sev = decide(p_label.ravel(), p_attack)
    t = y[split]
    return {"label acc": accuracy_score(t["label"], lab),
            "attack_cat acc": accuracy_score(t["attack_cat"], att),
            "attack_cat macro-F1": f1_score(t["attack_cat"], att, average="macro"),
            "attack_cat macro-precision": precision_score(t["attack_cat"], att, average="macro", zero_division=0),
            "attack_cat macro-recall": recall_score(t["attack_cat"], att, average="macro"),
            "severity acc": accuracy_score(t["severity"], sev),
            "severity macro-F1": f1_score(t["severity"], sev, average="macro")}, att

for p in POWERS:
    w = class_weights(y["train"]["attack_cat"], len(ATTACK_CLASSES), p)
    print(f"p = {p}:", {c: round(float(v), 2) for c, v in zip(ATTACK_CLASSES, w)})

## 5.2 Εκπαίδευση των τριών εκδοχών
Ίδιος σπόρος τυχαιότητας, ίδια αρχιτεκτονική, ίδιες ρυθμίσεις· αλλάζει **μόνο** το `p`.

In [ ]:
results_val, results_test, preds_test = {}, {}, {}
for p in POWERS:
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_model(X["train"].shape[1],
                        class_weights(y["train"]["attack_cat"], len(ATTACK_CLASSES), p),
                        class_weights(y["train"]["severity"], len(SEVERITY_CLASSES), p))
    history = model.fit(X["train"], targets("train"), validation_data=(X["val"], targets("val")),
                        epochs=60, batch_size=256, verbose=0,
                        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
                                   keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)])
    model.save(MODELS + f"unsw_multitask_p{p}.keras")
    results_val[f"p = {p}"], _ = evaluate(model, "val")
    results_test[f"p = {p}"], preds_test[p] = evaluate(model, "test")
    print(f"p = {p}: {len(history.history['loss'])} εποχές | val attack_cat acc = {results_val[f'p = {p}']['attack_cat acc']:.4f}, "
          f"macro-F1 = {results_val[f'p = {p}']['attack_cat macro-F1']:.4f}")

## 5.3 Σύγκριση
Πρώτα το **validation** (με αυτό διαλέγουμε), μετά το **test** (αυτό αναφέρουμε).

In [ ]:
val_table = pd.DataFrame(results_val).round(4)
test_table = pd.DataFrame(results_test).round(4)
print("VALIDATION"); print(val_table.to_string()); print()
print("TEST"); print(test_table.to_string())
pd.concat({"validation": val_table, "test": test_table}, axis=1).to_csv(MODELS + "unsw_class_weight_experiment.csv")

## 5.4 Τι αλλάζει ανά τύπο επίθεσης (test)
Precision και recall κάθε τύπου για τις τρεις εκδοχές, και πόσες φορές προβλέπεται κάθε τύπος σε σχέση με το πόσες φορές υπάρχει πραγματικά.

In [ ]:
labels = list(range(len(ATTACK_CLASSES)))
colors = {0.0: "#2a78d6", 0.25: "#eb6834", 0.5: "#1baf7a"}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
xpos = np.arange(len(ATTACK_CLASSES)); width = 0.27
for i, p in enumerate(POWERS):
    prec = precision_score(y["test"]["attack_cat"], preds_test[p], labels=labels, average=None, zero_division=0)
    rec = recall_score(y["test"]["attack_cat"], preds_test[p], labels=labels, average=None, zero_division=0)
    axes[0].bar(xpos + (i - 1) * width, prec, width * 0.92, color=colors[p], label=f"p = {p}")
    axes[1].bar(xpos + (i - 1) * width, rec, width * 0.92, color=colors[p], label=f"p = {p}")
for ax, t in zip(axes, ["Precision ανά τύπο (test)", "Recall ανά τύπο (test)"]):
    ax.set_xticks(xpos); ax.set_xticklabels(ATTACK_CLASSES, rotation=45, ha="right")
    ax.set_title(t); ax.set_ylim(0, 1.05); ax.grid(axis="y", alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(FIG + "22_unsw_class_weight_experiment.png", dpi=200); plt.show()

counts = pd.DataFrame({"πραγματικά": np.bincount(y["test"]["attack_cat"], minlength=len(ATTACK_CLASSES))}, index=ATTACK_CLASSES)
for p in POWERS:
    counts[f"προβλέψεις p = {p}"] = np.bincount(preds_test[p], minlength=len(ATTACK_CLASSES))
print(counts.to_string())